[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs_ar/Lab_5_NonHomogeneous_PDEs_AR.ipynb) [![Launch JupyterLab](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/hany73-max/PDE-Rover-Engine/main?urlpath=lab/tree/labs_ar/Lab_5_NonHomogeneous_PDEs_AR.ipynb)


In [ ]:
# 🚀 تهيئة مسار المشروع تلقائياً (يعمل على JupyterLab المحلي، وBinder، وGoogle Colab)
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        else:
            import subprocess
            subprocess.run(['git', '-C', 'PDE-Rover-Engine', 'pull', '--ff-only'], check=False)
        os.chdir('PDE-Rover-Engine')
except NameError:
    pass

# التأكد من إضافة المجلد الرئيسي للمشروع (الذي يحتوي على labs/ و rover_path_finder/) إلى مسار بايثون
# سواء تم تشغيل النوت بوك من المجلد الرئيسي أو من داخل labs_ar/ أو solutions/labs_ar/
_cwd = os.path.abspath(os.getcwd())
for _candidate in [_cwd, os.path.abspath(os.path.join(_cwd, '..')), os.path.abspath(os.path.join(_cwd, '..', '..'))]:
    if os.path.isdir(os.path.join(_candidate, 'labs', 'tests')) and os.path.isdir(os.path.join(_candidate, 'rover_path_finder')):
        if _candidate not in sys.path:
            sys.path.insert(0, _candidate)
        break
print('[OK] تم تهيئة مسار المشروع بنجاح!')


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

# ‏📡 اللاب 5: المعادلات الخطية غير المتجانسة في الرتبة، تحليل المؤثرات، واضمحلال الجهد في الممرات الضيقة

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏<i>غرفة التحكم: لماذا يفشل شرط ديريكليه (‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">dirichlet&#95;potential</code>‏) في الممرات الضيقة بسبب الاضمحلال الأسي؟</i>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المقرر:</b> المعادلات التفاضلية الجزئية وتطبيقاتها في الملاحة الفضائية (Applied PDE & Planetary Robotics)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المرجع في المذكرة:</b> الباب الثاني (Chapter 2)، القسم 2.1 — <b>ترقيم المذكرة (الباب الثاني) ص 1–10</b> <i>(صفحات ملف الـ PDF من 52 إلى 62؛ قانون العوامل الخطية في ص 2 / PDF ص 54، والمثال 5 في ص 4 / PDF ص 56)</i>

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏🗺️ موقعك في سلسلة بناء محرك الملاحة (Rover Pipeline)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; background-color:rgba(128,128,128,0.12); padding:10px 14px; border-radius:6px; overflow-x:auto; margin:10px 0; font-family:monospace; white-space:pre;">‏[اللاب 1–3: الميول، مسارات الانسياب، وهوامش الأمان] ──► [اللاب 4: معادلة لابلاس] ──► 📍 [اللاب 5: المؤثرات غير المتجانسة واضمحلال الجهد] ──► [اللاب 6: محرك الملاحة النهائي SOR + Neumann]</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏🧭 حجر رشيد (The Rosetta Stone): الربط بين السبورة ومحرك المركبة

</div>

<div dir="rtl" style="direction:rtl; text-align:right; overflow-x:auto; width:100%; margin:14px 0;">
<table dir="rtl" style="display:table; width:100%; border-collapse:collapse; direction:rtl; text-align:right;">
<thead>
<tr>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏1. رمز المذكرة (PDF pp. 52–62)

</th>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏2. المعنى الهندسي والفيزيائي

</th>
<th dir="rtl" align="right" style="direction:rtl; text-align:right; padding:9px 12px; border:1px solid rgba(128,128,128,0.4); background-color:rgba(128,128,128,0.16); font-weight:bold; line-height:1.75; vertical-align:middle;">

‏3. الدالة الفعلية في المشروع (‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">rover&#95;path&#95;finder/</code>‏)

</th>
</tr>
</thead>
<tbody>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${F(D_x, D_y)z = 0}$</span>‏ غير متجانسة في الرتبة

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏معادلة تجمع مشتقات من رتب مختلفة أو حداً ثابتاً ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${c}$</span>‏ (مثل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${aD_x + bD_y + c}$</span>‏)

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏تمثيل التخميد والاضمحلال الأسي على طول مسار الانتشار

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${(a D_x + b D_y + c)z = 0 \implies z = e^{-\frac{c}{a}x}\phi(ay - bx)}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏كل عامل خطي به ثابت ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${c \neq 0}$</span>‏ يفرض غلافاً أسياً ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${e^{-(c/a)x}}$</span>‏ على الحل

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏الاضمحلال الأسي للانحدار داخل الممرات الضيقة

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${(F_1 \cdot F_2)z = 0 \implies z = z_1 + z_2}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏تحليل المؤثر إلى عوامل خطية وجمع الحلول (مبدأ التراكب)

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏تحليل أنماط الجهد إلى مركبات أسية متراكبة

</td>
</tr>
<tr>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏تلاشي انحدار الجهد ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\Vert \nabla\phi\Vert  \propto e^{-\pi d / w}}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏عندما نجعل جدران العوائق ثابتة عند ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${\phi=1}$</span>‏، يتلاشى الميل أسياً داخل أي ممر عرضه ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; white-space:nowrap;">${w}$</span>‏

</td>
<td dir="rtl" align="right" style="direction:rtl; text-align:right; padding:8px 12px; border:1px solid rgba(128,128,128,0.35); vertical-align:middle; line-height:1.8;">

‏سبب استبدال <a href="rover_path_finder/planners.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">planners.dirichlet&#95;potential</code>‏</a> بالدالة <a href="rover_path_finder/planners.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">planners.neumann&#95;potential</code>‏</a>

</td>
</tr>
</tbody>
</table>
</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏1. أولاً: توضيح مصطلح مهم جداً في المذكرة ("غير متجانسة" في الباب الثاني!)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏عندما تقرأ عنوان <b>الباب الثاني في المذكرة (PDF ص 52): "Non-Homogeneous Linear PDEs"</b>، انتبه جيداً للفرق الاصطلاحي:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• في المعادلات العادية (ODEs)، كلمة "غير متجانسة" تعني عادةً وجود طرف أيمن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f(x, y) \neq 0}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• أما في <b>الباب الثاني من مذكرتنا (PDF ص 52–57)</b>، كلمة <b>Non-Homogeneous (غير متجانسة في الرتبة)</b> تعني أن <b>الطرف الأيسر نفسه يحتوي على مشتقات من رتب مختلفة</b> (مثلاً مشتقة ثانية ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x^2}$</span>‏ مع مشتقة أولى ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x}$</span>‏ أو ثابت ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${c}$</span>‏)، مثل:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$(2 D_x - 3 D_y + 1)(D_x + 2 D_y - 2)z = 0$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏لاحظ أن الطرف الأيمن يساوي صفراً، ومع ذلك تسمى المعادلة <b>غير متجانسة في الرتبة</b> بسبب وجود الثابتين ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${+1}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${-2}$</span>‏ مع المشتقات الأولى ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x, D_y}$</span>‏!

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏2. كيف يرتبط هذا الاضمحلال الأسي بمشكلة الممرات الضيقة في مركبة المريخ؟

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏في ملف <a href="rover_path_finder/planners.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">rover&#95;path&#95;finder/planners.py</code>‏</a>، توجد طريقتان لبناء حقل الجهد التوافقي للمركبة:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>1.</b> <b>الصياغة الأولى (‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">dirichlet&#95;potential</code>‏ — Formulation A):</b> تثبيت جميع العوائق وجدران الخريطة عند جهد مرتفع ثابت ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi = 1}$</span>‏، والهدف عند ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi = 0}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>2.</b> <b>الصياغة الثانية (‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">neumann&#95;potential</code>‏ — Formulation B):</b> جعل العوائق جدراناً عازلة (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\partial\phi/\partial n = 0}$</span>‏) ووضع مصدر عند البداية (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi=1}$</span>‏) ومصب عند الهدف (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi=0}$</span>‏).

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>لماذا تفشل الصياغة الأولى (‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">dirichlet&#95;potential</code>‏) عندما تمر المركبة في ممر ضيق بين صخرتين؟</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏إذا دخلت المركبة ممراً ضيقاً عرضه ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${w}$</span>‏ وطوله ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${d}$</span>‏ وجدرانه مثبتة عند ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi = 1}$</span>‏، فإن فرق الجهد ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\tilde{\phi} = 1 - \phi}$</span>‏ يحقق نمطاً توافقياً في اتجاه العرض ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\sin(\pi y / w)}$</span>‏ مما يفرض معادلة عاملية ذات حد ثابت ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${c = \pm\frac{\pi}{w}}$</span>‏ في اتجاه الطول ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\left(D_x + \frac{\pi}{w}\right)\tilde{\phi} = 0 \implies \tilde{\phi}(x, y) \propto e^{-\frac{\pi}{w} x} \sin\left(\frac{\pi y}{w}\right)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏أي أن انحدار الجهد يتلاشى أسياً بمعدل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${e^{-\pi x / w}}$</span>‏! فإذا كان الممر ضيقاً (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${w}$</span>‏ صغير)، يصبح الجهد ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\phi \approx 1.000000}$</span>‏ في كل مكان داخل الممر ويتوقف انحدار المركبة تماماً!

</div>

</div>


In [ ]:
import os, sys
for _p in ['.', '..', '../..']:
    _ap = os.path.abspath(_p)
    if os.path.isdir(os.path.join(_ap, 'labs', 'tests')) and _ap not in sys.path:
        sys.path.insert(0, _ap)
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab5

%matplotlib inline
print("تم تهيئة بيئة اللاب 5 بنجاح!")


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏3. الوصفة الرياضية الكاملة للباب الثاني في المذكرة (Cases A, B, C, D & P.I. — PDF ص 52–62)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏الحالة (A): عامل خطي واحد أو مكرر ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(a D_x + b D_y + c)^k z = 0}$</span>‏ <i>(الباب الثاني ص 2–3 / PDF ص 54–55)</i>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

#### ‏1. لماذا يظهر العامل الأسي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${e^{-\frac{c}{a}x}}$</span>‏؟

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• المعادلة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(a D_x + b D_y + c)z = 0}$</span>‏ هي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a \frac{\partial z}{\partial x} + b \frac{\partial z}{\partial y} = -c z}$</span>‏ (معادلة لاجرانج من اللاب 2 حيث ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${P=a, Q=b, R=-cz}$</span>‏!).

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• من معادلات لاجرانج المساعدة:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\frac{dx}{a} = \frac{dy}{b} = \frac{dz}{-cz}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• من الكسر الأول والثاني: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${b\,dx - a\,dy = 0 \implies ay - bx = u}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• من الكسر الأول والثالث (بشرط ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a \neq 0}$</span>‏): ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{dz}{z} = -\frac{c}{a}dx \implies \ln|z| = -\frac{c}{a}x + \ln|\phi(u)|}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• إذن <b>القانون المباشر لأي عامل خطي</b> هو:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>إذا كان ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a \neq 0}$</span>‏:</b>

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\boxed{(a D_x + b D_y + c)z = 0 \implies z = e^{-\frac{c}{a}x}\,\phi(a y - b x)}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>وإذا كان ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a = 0}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${b \neq 0}$</span>‏ (أو إذا اخترنا الكسر الثاني والثالث ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\frac{dy}{b} = \frac{dz}{-cz}}$</span>‏):</b>

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\boxed{z = e^{-\frac{c}{b}y}\,\phi(a y - b x)}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

#### ‏2. ماذا لو تكرر العامل الخطي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${k}$</span>‏ من المرات؟ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(a D_x + b D_y + c)^k z = 0}$</span>‏ (ص 2–3 / PDF ص 54–55)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>إذا استخدمنا الأس بدلالة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏ (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a \neq 0}$</span>‏):</b> نضرب الدوال المتتالية في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${1, x, x^2, \dots, x^{k-1}}$</span>‏:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\boxed{z = e^{-\frac{c}{a}x}\left[\phi_1(ay - bx) + x\,\phi_2(ay - bx) + x^2\,\phi_3(ay - bx) + \dots + x^{k-1}\phi_k(ay - bx)\right]}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>وإذا استخدمنا الأس بدلالة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${y}$</span>‏ (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${b \neq 0}$</span>‏، خاصة عندما ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a = 0}$</span>‏):</b> نضرب في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${1, y, y^2, \dots, y^{k-1}}$</span>‏:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\boxed{z = e^{-\frac{c}{b}y}\left[\phi_1(ay - bx) + y\,\phi_2(ay - bx) + \dots + y^{k-1}\phi_k(ay - bx)\right]}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; border-right:4px solid #f59e0b; background-color:rgba(245,158,11,0.08); padding:10px 14px; margin:12px 0; border-radius:4px; line-height:1.85;">

‏📌 <b>تنبيه خطأ مطبعي مهم جداً في المذكرة (الباب الثاني ص 2–3، المثالان 3 و 4 / PDF ص 54–55):</b><br>
‏* في <b>المثال 3 (PDF ص 54):</b> ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(2D_x - 3D_y + 2)^2 z = 0}$</span>‏ (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a=2, b=-3, c=2}$</span>‏)، الصيغة الأولى في المذكرة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z = e^{-x}[\phi_1(2y+3x) + x\phi_2(2y+3x)]}$</span>‏ <b>صحيحة 100%</b>! لكن في السطر الذي يليه ("Or ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z = e^{\frac{2}{3}y}[\dots]}$</span>‏")، كتبت المذكرة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x\phi_2}$</span>‏ بدلاً من ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${y\phi_2}$</span>‏.<br>
‏* وفي <b>المثال 4 (PDF ص 55):</b> ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x - 2D_y + 1)^4 z = 0}$</span>‏ (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a=1, b=-2, c=1, k=4}$</span>‏)، وضعت المذكرة داخل القوس قوى ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏ (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${1, x, x^2, x^3}$</span>‏) ولكنها كتبت خارج القوس بالخطأ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${e^{\frac{1}{2}y}}$</span>‏ بدلاً من ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${e^{-\frac{c}{a}x} = e^{-x}}$</span>‏!<br>
‏* <b>الخلاصة الذهبية لك في الامتحان:</b> التزم دائماً بصيغة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏ القياسية (طالما ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a \neq 0}$</span>‏): <b>أخرج ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${e^{-(c/a)x}}$</span>‏ واضرب بالداخل في قوى ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${x}$</span>‏ (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${1, x, x^2, \dots}$</span>‏)</b>، فهي الأسهل والأضمن 100%!

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏الحالة (B): المؤثرات القابلة للتحليل إلى عوامل خطية مختلفة <i>(الباب الثاني ص 3–4 / PDF ص 55–56)</i>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏إذا أمكن تحليل المؤثر ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${F(D_x, D_y)}$</span>‏ إلى حاصل ضرب عوامل خطية:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$(a_1 D_x + b_1 D_y + c_1)(a_2 D_x + b_2 D_y + c_2)z = 0$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏فإن الحل العام هو مجموع حلول كل عامل على حدة:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$z = e^{-\frac{c_1}{a_1}x}\phi_1(a_1 y - b_1 x) + e^{-\frac{c_2}{a_2}x}\phi_2(a_2 y - b_2 x)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

#### ‏📖 مثال محلول بالتفصيل (المثال 5، الباب الثاني ص 4 / PDF ص 56)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المسألة:</b> أوجد الحل العام للمعادلة:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$(2 D_x - 3 D_y + 1)(D_x + 2 D_y - 2)z = 0$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>1.</b> <b>العامل الأول ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(2 D_x - 3 D_y + 1)z = 0}$</span>‏:</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ المعاملات: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a_1 = 2,\; b_1 = -3,\; c_1 = 1}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ الأس: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${-\frac{c_1}{a_1}x = -\frac{1}{2}x}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ داخل الدالة الاختيارية: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a_1 y - b_1 x = 2y - (-3)x = 2y + 3x}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ إذن الحل الأول: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z_1 = e^{-x/2}\Phi_1(2y + 3x)}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>2.</b> <b>العامل الثاني ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x + 2 D_y - 2)z = 0}$</span>‏:</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ المعاملات: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a_2 = 1,\; b_2 = 2,\; c_2 = -2}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ الأس: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${-\frac{c_2}{a_2}x = -\frac{-2}{1}x = 2x}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ داخل الدالة الاختيارية: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a_2 y - b_2 x = 1y - 2x = y - 2x}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ إذن الحل الثاني: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z_2 = e^{2x}\Phi_2(y - 2x)}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>3.</b> <b>الحل العام الكلي:</b> ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\boxed{z = e^{-x/2}\Phi_1(2y + 3x) + e^{2x}\Phi_2(y - 2x)}}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<i>(وكذلك في <b>المثال 7 ص 5 / PDF ص 57</b>: المعادلة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x^2 - D_y^2 - 3D_x + 3D_y)z = 0}$</span>‏ نحللها بالتجميع أولاً: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x - D_y)(D_x + D_y) - 3(D_x - D_y) = (D_x - D_y)(D_x + D_y - 3)z = 0 \implies \boxed{z = \phi_1(y + x) + e^{3x}\phi_2(y - x)}}$</span>‏!)</i>

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏الحالتان (C) و (D): المؤثرات غير القابلة للتحليل (Irreducible Operators — الباب الثاني ص 5–8 / PDF ص 57–60)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>متى نستخدم الحالة (C)؟</b> عندما لا يمكن تحليل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${F(D_x, D_y)}$</span>‏ إلى عوامل خطية من الدرجة الأولى، مثل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x - D_y^3)z = 0}$</span>‏ أو ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x^2 + D_x + D_y)z = 0}$</span>‏ أو معادلة الحرارة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x^2 - D_y)z = 0}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>قاعدة الحل في المذكرة (ص 5–6 / PDF ص 57–58):</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏نفرض الحل في صورة متسلسلة من الدوال الأسية:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\boxed{z = \sum_{i=1}^{\infty} A_i\, e^{a_i x + b_i y} \qquad \text{بشرط أن تحقق } (a_i, b_i)\text{ المعادلة المميزة: } F(a_i, b_i) = 0}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏حيث ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${A_i}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a_i}$</span>‏ (أو ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${b_i}$</span>‏) ثوابت اختيارية!

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>📖 الأمثلة المحلولة في المذكرة (أمثلة 8 و 9 و 11 — ص 6–7 / PDF ص 58–59):</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:26px; line-height:1.85;">

‏<b>1.</b> <b>مثال 8:</b> حل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x - D_y^3)z = 0}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ المؤثر غير قابل للتحليل لأن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x}$</span>‏ من الرتبة 1 و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_y^3}$</span>‏ من الرتبة 3!

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ نضع ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x \to a_i}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_y \to b_i}$</span>‏ في الشرط ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${F(a_i, b_i) = 0 \implies a_i - b_i^3 = 0 \implies \boxed{a_i = b_i^3}}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ نعوض عن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a_i}$</span>‏ بـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${b_i^3}$</span>‏ في المتسلسلة الأسية:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\boxed{z = \sum_{i=1}^{\infty} A_i\, e^{b_i^3 x + b_i y}}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:26px; line-height:1.85;">

‏<b>2.</b> <b>مثال 9:</b> حل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x^2 + D_x + D_y)z = 0}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ الشرط المميز: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a_i^2 + a_i + b_i = 0 \implies \boxed{b_i = -a_i^2 - a_i = -a_i(a_i + 1)}}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ الحل العام: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\boxed{z = \sum_{i=1}^{\infty} A_i\, e^{a_i x - a_i(a_i + 1)y}}}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:26px; line-height:1.85;">

‏<b>3.</b> <b>الحالة (D) — مزيج من عامل خطي قابل للتحليل وعامل غير قابل للتحليل (مثال 11 ص 7 / PDF ص 59):</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x - D_y - 1)(D_x - D_y^2)z = 0}$</span>‏:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ العامل الأول خطي يعطي: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z^{(1)} = e^x \phi(y + x)}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ العامل الثاني ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x - D_y^2)}$</span>‏ غير قابل للتحليل (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a_i = b_i^2}$</span>‏) يعطي: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z^{(2)} = \sum_{i=1}^{\infty} A_i e^{b_i^2 x + b_i y}}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ نجمعهما معاً: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\boxed{z = e^x \phi(y + x) + \sum_{i=1}^{\infty} A_i e^{b_i^2 x + b_i y}}}$</span>‏!

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏رابعاً: التكامل الخاص (P.I.) في الباب الثاني وحيلة "المرافق التفاضلي" (الباب الثاني ص 8–10 / PDF ص 60–62)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏لإيجاد التكامل الخاص ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z_2 = \frac{1}{F(D_x, D_y)}f(x, y)}$</span>‏ للمعادلات غير المتجانسة في الرتبة، نستخدم نفس قواعد اللاب 4 مع <b>مهارتين مهمتين جداً</b> في نهاية المذكرة:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

#### ‏1. فك كثيرات الحدود بسحب أصغر مشتقة (مثال 12 ص 8 / PDF ص 60)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>المسألة:</b> أوجد الحل العام لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x^2 - D_y^2 + D_x)z = x^2 + 2y}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ لحساب التكامل الخاص ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z_2 = \frac{1}{D_x^2 - D_y^2 + D_x}(x^2 + 2y)}$</span>‏، نسحب <b>أصغر مشتقة وهي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x}$</span>‏</b> من المقام:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$z_2 = \frac{1}{D_x\left[1 - \left(\frac{D_y^2}{D_x} - D_x\right)\right]}(x^2 + 2y) = \frac{1}{D_x}\left[1 + \left(\frac{D_y^2}{D_x} - D_x\right) + \left(\frac{D_y^2}{D_x} - D_x\right)^2 + \dots\right](x^2 + 2y)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ بما أن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_y^2(x^2 + 2y) = 0}$</span>‏، تختفي كل حدود ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_y^2}$</span>‏! ويبقى فقط:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$z_2 = \frac{1}{D_x}\left(1 - D_x + D_x^2\right)(x^2 + 2y) = \frac{1}{D_x}\left[(x^2 + 2y) - (2x) + 2\right] = \boxed{\frac{x^3}{3} + 2xy - x^2 + 2x}$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

#### ‏2. حيلة الضرب في المرافق لدوال ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\sin}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\cos}$</span>‏ عندما تبقى مشتقة من الرتبة الأولى! (مثال 14 ص 9–10 / PDF ص 61–62)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:14px; line-height:1.85;">

‏• <b>المسألة:</b> أوجد الحل العام لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x^2 - D_y)z = \sin(x - 2y)}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:26px; line-height:1.85;">

‏<b>1.</b> <b>الحل المكمل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z_1}$</span>‏:</b> المؤثر ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x^2 - D_y)}$</span>‏ غير قابل للتحليل (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a_i^2 - b_i = 0 \implies b_i = a_i^2}$</span>‏)، إذن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z_1 = \sum_{i=1}^{\infty} A_i e^{a_i x + a_i^2 y}}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:26px; line-height:1.85;">

‏<b>2.</b> <b>التكامل الخاص ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z_2}$</span>‏:</b> هنا ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${f(x,y) = \sin(x - 2y)}$</span>‏ حيث ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a = 1,\; b = -2}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ قاعدة الجيب تستبدل المشتقات <b>من الرتبة الثانية فقط</b> (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x^2 \to -a^2 = -1}$</span>‏ و ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_y^2 \to -b^2 = -4}$</span>‏)، ولكن المقام هنا هو ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x^2 - D_y)}$</span>‏ فيحتوي على ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_y}$</span>‏ من الرتبة الأولى!

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>الخطوة 1:</b> عوض عن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x^2 \to -1}$</span>‏ أولاً:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$z_2 = \frac{1}{-1 - D_y}\sin(x - 2y)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>الخطوة 2 (حيلة المرافق!):</b> كيف نتخلص من ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_y}$</span>‏ في المقام ونحولها إلى ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_y^2}$</span>‏؟ نضرب البسط والمقام في <b>المرافق ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(-1 + D_y)}$</span>‏</b>!

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$z_2 = \frac{-1 + D_y}{(-1 - D_y)(-1 + D_y)}\sin(x - 2y) = \frac{-1 + D_y}{1 - D_y^2}\sin(x - 2y)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>الخطوة 3:</b> الآن ظهر ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_y^2}$</span>‏ في المقام! نعوض عنه بـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${-b^2 = -(-2)^2 = -4}$</span>‏:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$z_2 = \frac{-1 + D_y}{1 - (-4)}\sin(x - 2y) = \frac{1}{5}(-1 + D_y)\sin(x - 2y)$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:5px 0; padding-right:34px; line-height:1.85;">

‏▪ <b>الخطوة 4:</b> الآن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(-1 + D_y)}$</span>‏ في البسط يعني ببساطة: اضرب في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${-1}$</span>‏ ثم اجمع المشتقة الجزئية بالنسبة لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${y}$</span>‏ (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_y[\sin(x-2y)] = -2\cos(x-2y)}$</span>‏):

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$\boxed{z_2 = \frac{1}{5}\big[-\sin(x - 2y) - 2\cos(x - 2y)\big]}$$

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

### ‏⚠️ أخطاء شائعة في الإشارات يجب الحذر منها

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>1.</b> <b>نسيان إشارة السالب في الأس ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${-\frac{c}{a}x}$</span>‏:</b> إذا كان ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${c}$</span>‏ موجباً (مثل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${+1}$</span>‏)، يكون الأس سالباً (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${-x/a}$</span>‏)؛ وإذا كان ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${c}$</span>‏ سالباً (مثل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${-2}$</span>‏)، يصبح الأس موجباً (‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${+2x/a}$</span>‏).

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>2.</b> <b>نسيان إشارة السالب في ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(ay - bx)}$</span>‏:</b> القانون يحتوي على طرح ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${-bx}$</span>‏! فإذا كان ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${b = -3}$</span>‏، تصبح ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${ay - (-3)x = ay + 3x}$</span>‏.

</div>

</div>


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏4. 📝 تحدي الورقة والقلم 1 (المثال 5 من الباب الثاني ص 4 / PDF ص 56)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المسألة:</b> أوجد الحل العام للمعادلة التفاضلية الجزئية القابلة للتحليل:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$(2 D_x - 3 D_y + 1)(D_x + 2 D_y - 2)z = 0$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏اكتب صيغة الحل العام بدلالة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">x, y</code>‏ والدالتين الاختياريتين ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">Phi1(...)</code>‏ و ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">Phi2(...)</code>‏ والدالة الأسية ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">exp(...)</code>‏:

</div>

<details dir="rtl" style="direction:rtl; text-align:right; margin:8px 0; padding:6px 10px;">

<summary dir="rtl" style="direction:rtl; text-align:right; cursor:pointer; font-weight:bold;">

‏💡 <b>تلميح 1 (العامل الأول)</b>

</summary>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏للعامل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(2 D_x - 3 D_y + 1)}$</span>‏: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a_1=2, b_1=-3, c_1=1 \implies \text{exp}(-x/2)\text{*Phi1}(2y + 3x)}$</span>‏.

</div>

</details>

<details dir="rtl" style="direction:rtl; text-align:right; margin:8px 0; padding:6px 10px;">

<summary dir="rtl" style="direction:rtl; text-align:right; cursor:pointer; font-weight:bold;">

‏💡 <b>تلميح 2 (العامل الثاني والجمع)</b>

</summary>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏للعامل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(D_x + 2 D_y - 2)}$</span>‏: ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a_2=1, b_2=2, c_2=-2 \implies \text{exp}(2x)\text{*Phi2}(y - 2x)}$</span>‏. اجمع الحدين معاً.

</div>

</details>

</div>


In [ ]:
# خلية تقييم: تحدي الورقة 1
### START CODE HERE ### (≈ 1 line of code)
# اكتب الحل العام بدلالة exp(...) و Phi1(...) و Phi2(...):
student_solution = None
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab5.check_paper_challenge_1(student_solution)


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏5. 🚀 تحدي الورقة والقلم 2 (حساب معدل الاضمحلال الأسي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${k = -c/a}$</span>‏)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>المسألة:</b> لنفرض أن نمطاً موجياً يخضع للمعادلة العاملية الخطية:

</div>

<div dir="ltr" align="center" style="direction:ltr; text-align:center; margin:12px 0; overflow-x:auto;">

$$(D_x + 2 D_y + 1)z = 0$$

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏<b>على ورقتك:</b>

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>1.</b> استخرج المعاملات ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a, b, c}$</span>‏ من العامل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(a D_x + b D_y + c)}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>2.</b> في الحل العام ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z = e^{k x} \Phi(a y - b x)}$</span>‏، يُعطى معدل التخميد الأسي بالعلاقة ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${k = -\frac{c}{a}}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>3.</b> احسب القيمة العددية لـ ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${k}$</span>‏ وأدخلها أدناه:

</div>

<details dir="rtl" style="direction:rtl; text-align:right; margin:8px 0; padding:6px 10px;">

<summary dir="rtl" style="direction:rtl; text-align:right; cursor:pointer; font-weight:bold;">

‏💡 <b>تلميح (اضغط هنا إذا احتجت مساعدة)</b>

</summary>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏معامل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${D_x}$</span>‏ هو ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${a = 1}$</span>‏، والحد الثابت هو ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${c = 1}$</span>‏. إذن ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${k = -c/a = -1/1}$</span>‏.

</div>

</details>

</div>


In [ ]:
# خلية تقييم: معدل الاضمحلال الأسي
### START CODE HERE ### (≈ 1 line of code)
# أدخل القيمة العددية لمعدل الاضمحلال الأسي k = -c/a:
damping_rate = None
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab5.check_paper_challenge_2(damping_rate)


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏6. 💻 التطبيق البرمجي ببايثون: محلل معاملات العوامل الخطية (‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">parse&#95;linear&#95;factor</code>‏)

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏نريد كتابة دالة برمجية تستقبل نص عامل خطي مثل ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">"2&#42;D&#95;x - 3&#42;D&#95;y - 5"</code>‏ وتستخرج منه المعاملات العددية الثلاثة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">(a, b, c)</code>‏ تلقائياً لحساب معدل الاضمحلال الأسي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${-c/a}$</span>‏ وزاوية الانتشار ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(a, -b)}$</span>‏.

</div>

</div>


In [ ]:
# دالة تقييم برمجية: parse_linear_factor
def parse_linear_factor(factor_str: str) -> tuple[float, float, float]:
    # تستخرج المعاملات (a, b, c) من نص العامل الخطي (مثال: "2*D_x - 3*D_y - 5" -> (2.0, -3.0, -5.0))
    ### START CODE HERE ### (≈ 8-12 lines of code)
    import re
    # يمكنك استخدام التعبيرات النمطية (re.search) أو تحليل السلسلة النصية لاستخراج a, b, c:
    a = None
    b = None
    c = None
    ### END CODE HERE ###
    return a, b, c

# تشغيل الاختبار الآلي:
test_lab5.test_linear_factor_evaluator(parse_linear_factor)


<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; width:100%; max-width:100%; box-sizing:border-box; line-height:1.85; padding:6px 14px 16px 14px;">

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:12px 0 6px 0;">

## ‏7. 🎨 الرؤية البصرية والربط المباشر بملف ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">rover&#95;path&#95;finder/planners.py</code>‏

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:6px 0; line-height:1.85;">

‏شغّل الخلية التالية لترى:

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>1.</b> <b>على اليسار:</b> السطح الأسي المتلاشي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${z = e^{-0.5x}\sin(y - 2x)}$</span>‏ الناتج عن العامل الخطي ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${(2D_x + D_y + 1)z = 0}$</span>‏.

</div>

<div dir="rtl" align="right" style="direction:rtl; text-align:right; unicode-bidi:embed; margin:7px 0; padding-right:8px; line-height:1.85;">

‏<b>2.</b> <b>على اليمين:</b> مقارنة حقيقية من كود المشروع <a href="rover_path_finder/planners.py">‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">rover&#95;path&#95;finder/planners.py</code>‏</a> بين دالة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">dirichlet&#95;potential</code>‏ (التي ينهار فيها انحدار الجهد أسياً بمعدل ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${e^{-\pi x / w}}$</span>‏ داخل ممر ضيق فتتوقف المركبة!) ودالة ‏<code dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate; padding:1px 5px; border-radius:3px;">neumann&#95;potential</code>‏ (التي تعزل جدران الممر ‏<span dir="ltr" style="direction:ltr; display:inline-block; unicode-bidi:isolate;">${\partial\phi/\partial n = 0}$</span>‏ فينحدر الجهد خطياً بسلاسة من البداية إلى الهدف!).

</div>

</div>


In [ ]:
from rover_path_finder.planners import dirichlet_potential, neumann_potential

# 1. السطح التحليلي للعامل الخطي (2 D_x + D_y + 1)z = 0
x = np.linspace(0, 5, 50)
y = np.linspace(-3, 3, 50)
X, Y = np.meshgrid(x, y)
Z = np.exp(-0.5 * X) * np.sin(Y - 2.0 * X)

# 2. تجربة ممر ضيق بين جدارين باستخدام دوال المشروع الفعلية
corridor_hazard = np.zeros((40, 60), dtype=bool)
corridor_hazard[:16, 15:45] = True   # الجدار العلوي للممر الضيق
corridor_hazard[24:, 15:45] = True   # الجدار السفلي للممر الضيق (عرض الممر w = 8 بكسل)
start_pt, goal_pt = (8, 20), (52, 20)

phi_dir, _, _ = dirichlet_potential(corridor_hazard, goal=goal_pt, tol=1e-8)
phi_neu, _, _ = neumann_potential(corridor_hazard, start=start_pt, goal=goal_pt, tol=1e-6)

fig = plt.figure(figsize=(14, 5.5))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
surf = ax1.plot_surface(X, Y, Z, cmap='plasma', alpha=0.88)
ax1.set_title(r"1. Linear Factor Solution: $z = e^{-0.5x}\sin(y - 2x)$", fontsize=11)
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.set_zlabel("z")
plt.colorbar(surf, ax=ax1, shrink=0.55)

ax2 = fig.add_subplot(1, 2, 2)
center_x = np.arange(8, 53)
ax2.plot(center_x, 1.0 - phi_dir[20, 8:53], 'r--', linewidth=2.5, label=r"1 - dirichlet_potential (Exponential Decay $e^{-\pi x/w}$)")
ax2.plot(center_x, phi_neu[20, 8:53], 'b-', linewidth=2.5, label="neumann_potential (Smooth Linear Gradient)")
ax2.set_yscale('log')
ax2.set_title("2. Real Engine: Narrow Corridor Gradient Decay (Log Scale)", fontsize=11)
ax2.set_xlabel("Position along narrow corridor (x)")
ax2.set_ylabel("Potential Drop (log scale)")
ax2.grid(True, which="both", alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

print("📌 ملخص اللاب 5:")
print("  1) العامل الخطي (a*D_x + b*D_y + c)z = 0 ينتج حلاً مضروباً في غلاف أسي exp(-(c/a)*x).")
print("  2) في الممرات الضيقة، شرط ديريكليه على الجدران يخلق تلاشيًا أسيًا exp(-pi*x/w) يوقف المركبة.")
print("  3) لهذا السبب يستخدم محرك الملاحة في اللاب 6 شروط نيومان العازلة (neumann_potential) بدلاً من ديريكليه!")
